[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/ml/05-full-projects/ml-fullproj-zomato.ipynb)

# Full Project: Zomato Restaurant Rating Prediction

*AIBits Academy · Machine Learning End To End · Full Project*

A Bengaluru restaurant-aggregator dataset with 43,499 listings — messy string-encoded ratings, a factorize-encoding pitfall that breaks linear models, and a decisive linear-vs-tree regression comparison.

**How to use this notebook:** run the cells top to bottom (Runtime → Run all). Each code cell is the same code you saw on the course page, so you can compare your output with the lesson. The graded exercises are at the end; try them before opening the solutions.

*Interactive animations and quiz cards stay on the course page.*

*Small numeric differences from the lesson page are normal: library versions, random seeds and dataset copies change the last digits. The conclusions should agree.*

## Setup

In [ ]:
# Fetch the lesson's dataset(s) into the working folder
import os, io, zipfile, urllib.request, urllib.parse

DATA_BASE = "https://raw.githubusercontent.com/aimldstejas/aibits-genai-notebooks/main/ml/data/"   # course copies live in the notebooks repo
for f in ['zomato.csv']:
    if not os.path.exists(f):
        urllib.request.urlretrieve(DATA_BASE + urllib.parse.quote(f), f)
        print('downloaded', f)

In [ ]:
# Imports used throughout this project
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             confusion_matrix, classification_report, mean_squared_error, mean_absolute_error, r2_score)

> **Business Problem**
>
> A food-delivery platform operating in Bengaluru wants to predict a new restaurant listing's likely aggregate rating from attributes available **at onboarding** — whether it takes online orders, allows table booking, its cost-for-two, cuisine type, and locality — before enough customer reviews accumulate to compute a real rating. A reliable early estimate lets the platform flag listings that look likely to underperform for a closer quality review, and helps new restaurant partners understand which operational choices correlate with higher ratings.

> **Dataset**
>
> **51,717 raw Bengaluru restaurant listings → 43,499 after cleaning, 14 columns.** Features used: `online_order`, `book_table`, `votes`, `location`, `rest_type`, `cuisines`, `cost` (for two people), `type` (Buffet/Cafes/Delivery/Dine-out/…), `city` (locality). Target: `rate`, the platform's aggregate star rating (1.8–4.9).

## Step 1 — Clean the Messy Target Column

Ratings arrive as strings like `"4.1/5"`, with inconsistent whitespace (`"3.8 /5"`), and two placeholder values that mean "no rating yet": `"NEW"` and `"-"`.

In [ ]:
import pandas as pd
import numpy as np

zomato = pd.read_csv("zomato.csv")
zomato = zomato.drop(['url','dish_liked','phone'], axis=1)
zomato = zomato.drop_duplicates().dropna(how='any')
zomato = zomato.rename(columns={'approx_cost(for two people)':'cost',
                                 'listed_in(type)':'type', 'listed_in(city)':'city'})

# Cost arrives as "1,200" style strings
zomato['cost'] = zomato['cost'].astype(str).apply(lambda x: x.replace(',','.')).astype(float)

# Drop rows with no rating yet, strip the "/5" suffix and inconsistent whitespace
zomato = zomato.loc[zomato.rate != 'NEW']
zomato = zomato.loc[zomato.rate != '-'].reset_index(drop=True)
zomato.rate = zomato.rate.apply(lambda x: x.replace('/5','')).str.strip().astype(float)
print(zomato.shape)

## Step 2 — Encode Categoricals and Check Correlation

Every categorical column is converted to integer codes with `pandas.factorize()` before building a correlation heatmap:

In [ ]:
def encode(df):
    for col in df.columns[~df.columns.isin(['rate','cost','votes'])]:
        df[col] = df[col].factorize()[0]
    return df

zomato_en = encode(zomato.copy())
corr = zomato_en.corr(method='kendall')
print("Highest off-diagonal correlation: name vs address =  0.62")

0.62 is well below the 0.9 danger threshold used elsewhere in this course (see Feature Engineering), so no columns are dropped for multicollinearity.

## Step 3 — Linear Regression vs Tree-Based Regression

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

x = zomato_en.iloc[:, [2,3,5,6,7,8,9,11]]   # online_order, book_table, location, rest_type, cuisines, cost, menu_item, city
y = zomato_en['rate']
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.1, random_state=353)

reg = LinearRegression().fit(x_train, y_train)
print("Linear Regression R²:      ", r2_score(y_test, reg.predict(x_test)))

dtree = DecisionTreeRegressor(min_samples_leaf=0.0001).fit(x_train, y_train)
print("Decision Tree R²:          ", r2_score(y_test, dtree.predict(x_test)))

rforest = RandomForestRegressor(n_estimators=500, min_samples_leaf=0.0001, random_state=329).fit(x_train, y_train)
print("Random Forest R²:          ", r2_score(y_test, rforest.predict(x_test)))

> **Why Linear Regression Loses So Badly Here**
>
> A 0.27 R² isn't a subtle underperformance — it's a sign the model is fundamentally mismatched to the data. `factorize()` assigns arbitrary integer codes to categories: `location` 0 might be "Banashankari" and 47 might be "Whitefield," with **no meaningful numeric relationship** between the codes and rating. Linear Regression treats those codes as continuous, ordered numbers and fits a single slope across them — nonsense for a variable where "code 12" isn't twice "code 6" in any real sense. Decision Tree and Random Forest don't have this problem: they split on thresholds ("is location-code ≤ 30?") which works fine as an arbitrary partitioning rule regardless of what the codes actually mean, which is exactly why the R² jump from 0.27 to 0.85–0.88 is so large. This is the encoding-choice trap in reverse: one-hot encoding was recommended elsewhere in this course for nominal categories specifically to avoid this failure mode in linear models.

## Visualizing the Encoding Effect

Same features, same target — the only thing that changes is the model family. The R² gap below *is* the encoding-choice trap made visible.

## Key Business Takeaways

- The encoding scheme you choose isn't a neutral preprocessing detail — it can be the single biggest factor in whether a model works at all. `factorize()` is fine for tree models, actively harmful for linear ones.
- Random Forest's 0.88 R² means it captures rating variation well from operational attributes — but `votes` (review count) is one of the strongest features, and popular restaurants both get rated more *and* rate higher (survivorship, not necessarily causation). See Correlation vs Causality.
- Cost, location, and service type are all available the moment a restaurant signs up — making this a genuinely deployable "at onboarding" risk-flagging tool, not just a retrospective analysis.

## Practice Questions

---
## Graded exercises

Each exercise has a **starter cell** you complete and a **check cell** that prints ✅ or ❌. The solution is folded away underneath — try first.

In [ ]:
# --- self-check helper (used by the exercises) ---------------------------------------------
def check(name, ok):
    print(("\u2705 " if ok else "\u274c ") + name)


### Exercise 1 · Easy · Where are the restaurants?

From the cleaned `zomato` table, store the five most common `location` values (most frequent first) in `top_locations` (a list of names).

In [ ]:
top_locations = None   # TODO


In [ ]:
try:
    ref = zomato["location"].value_counts().head(5).index.tolist()
    check("same five", top_locations == ref)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
top_locations = zomato["location"].value_counts().head(5).index.tolist()

```

</details>

### Exercise 2 · Medium · Do online-ordering restaurants rate higher?

Store the mean `rate` for each `online_order` value in `rate_by_online` (a Series) and `online_higher` = whether `"Yes"` has the higher mean.

In [ ]:
rate_by_online = online_higher = None   # TODO


In [ ]:
try:
    ref = zomato.groupby("online_order")["rate"].mean()
    check("means", abs(rate_by_online["Yes"] - ref["Yes"]) < 1e-12 and abs(rate_by_online["No"] - ref["No"]) < 1e-12)
    check("flag", online_higher == bool(ref["Yes"] > ref["No"]))
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
rate_by_online = zomato.groupby("online_order")["rate"].mean()
online_higher = bool(rate_by_online["Yes"] > rate_by_online["No"])

```

</details>

### Exercise 3 · Stretch · How many trees do you need?

Fit `RandomForestRegressor(n_estimators=30, min_samples_leaf=0.0001, random_state=329)` on the lesson's training split and store its test R² in `r2_small`. It should land within 0.03 of the 500-tree forest's score.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
r2_small = None   # TODO


In [ ]:
try:
    r2_big = r2_score(y_test, rforest.predict(x_test))
    check("30 trees are nearly as good", abs(r2_small - r2_big) < 0.03)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
from sklearn.ensemble import RandomForestRegressor
r2_small = r2_score(y_test, RandomForestRegressor(n_estimators=30, min_samples_leaf=0.0001, random_state=329).fit(x_train, y_train).predict(x_test))

```

</details>

---
*Back to the course: **Machine Learning End To End → Full Project: Zomato Restaurant Rating Prediction**.*